# Sistema de Scoring Crediticio con Optimización de Rentabilidad Ajustada por Riesgo

In [118]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [119]:
df = pd.read_csv(r"C:\Users\camil\Documents\Laburo\Portafolio\credit-risk\data\raw\accepted_2007_to_2018Q4.csv", low_memory= False)

In [120]:
dict_df = pd.read_excel(r"C:\Users\camil\Documents\Laburo\Portafolio\credit-risk\data\raw\LCDataDictionary.xlsx")

c:\Users\camil\anaconda3\envs\mlp311\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [121]:
browse_feat = dict_df["LoanStatNew"].dropna().values

In [122]:
data_feat = df.columns

print("En Browse pero no en data:")
print(set(browse_feat) - set(data_feat))

print("\nEn data pero no en Browse:")
print(set(data_feat) - set(browse_feat))

En Browse pero no en data:
{'open_il_6m', 'total_rev_hi_lim \xa0', 'is_inc_v', 'verified_status_joint'}

En data pero no en Browse:
{'num_accts_ever_120_pd', 'total_rev_hi_lim', 'pub_rec_bankruptcies', 'avg_cur_bal', 'acc_open_past_24mths', 'disbursement_method', 'sec_app_open_act_il', 'mo_sin_old_il_acct', 'tax_liens', 'sec_app_mths_since_last_major_derog', 'orig_projected_additional_accrued_interest', 'mths_since_recent_bc', 'num_tl_op_past_12m', 'num_rev_accts', 'num_bc_tl', 'total_bc_limit', 'num_actv_rev_tl', 'bc_open_to_buy', 'sec_app_open_acc', 'num_tl_90g_dpd_24m', 'pct_tl_nvr_dlq', 'hardship_flag', 'hardship_end_date', 'hardship_payoff_balance_amount', 'num_rev_tl_bal_gt_0', 'sec_app_num_rev_accts', 'hardship_dpd', 'hardship_loan_status', 'num_op_rev_tl', 'hardship_type', 'settlement_date', 'mo_sin_rcnt_tl', 'hardship_start_date', 'verification_status_joint', 'num_sats', 'mths_since_recent_bc_dlq', 'sec_app_mort_acc', 'chargeoff_within_12_mths', 'settlement_status', 'num_tl_12

In [123]:
browse_feat = [x.replace("\xa0", "") for x in browse_feat]

correcciones = {
    "is_inc_v": "verification_status",
    "verified_status_joint": "verification_status_joint"
}

browse_feat = [correcciones.get(col, col) for col in browse_feat]

In [124]:
valid_features = list(set(browse_feat).intersection(set(df.columns)))

In [125]:
len(valid_features)

76

In [126]:
df_browse = df[valid_features].copy()
df_browse["loan_status"] = df["loan_status"]

In [127]:
df_browse.shape

(2260701, 76)

In [128]:
df_browse.head()

,mths_since_last_major_derog,open_acc,acc_now_delinq,int_rate,title,emp_length,fico_range_high,fico_range_low,pub_rec,installment,...,open_il_12m,last_pymnt_d,sub_grade,inq_last_6mths,revol_util,earliest_cr_line,annual_inc_joint,last_credit_pull_d,grade,addr_state
0,30.0,7.0,0.0,13.99,Debt consolidation,10+ years,679.0,675.0,0.0,123.03,...,0.0,Jan-2019,C4,1.0,29.7,Aug-2003,NaN,Mar-2019,C,PA
1,NaN,22.0,0.0,11.99,Business,10+ years,719.0,715.0,0.0,820.28,...,0.0,Jun-2016,C1,4.0,19.2,Dec-1999,NaN,Mar-2019,C,SD
2,NaN,6.0,0.0,10.78,NaN,10+ years,699.0,695.0,0.0,432.66,...,0.0,Jun-2017,B4,0.0,56.2,Aug-2000,71000.0,Mar-2019,B,IL
3,NaN,13.0,0.0,14.85,Debt consolidation,10+ years,789.0,785.0,0.0,829.90,...,0.0,Feb-2019,C5,0.0,11.6,Sep-2008,NaN,Mar-2019,C,NJ
4,NaN,12.0,0.0,22.45,Major purchase,3 years,699.0,695.0,0.0,289.91,...,0.0,Jul-2016,F1,3.0,64.5,Jun-1998,NaN,Mar-2018,F,PA


In [129]:
df_browse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 76 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   mths_since_last_major_derog  float64
 1   open_acc                     float64
 2   acc_now_delinq               float64
 3   int_rate                     float64
 4   title                        object 
 5   emp_length                   object 
 6   fico_range_high              float64
 7   fico_range_low               float64
 8   pub_rec                      float64
 9   installment                  float64
 10  tot_coll_amt                 float64
 11  annual_inc                   float64
 12  total_pymnt_inv              float64
 13  verification_status          object 
 14  mths_since_last_delinq       float64
 15  policy_code                  float64
 16  emp_title                    object 
 17  application_type             object 
 18  term                         object 
 19  

In [130]:
print(f'The dataset has {df_browse.shape[0]} rows and {df_browse.shape[1]} columns.')

The dataset has 2260701 rows and 76 columns.


In [132]:
missing_table = pd.DataFrame({
    "Valores_Faltantes": df_browse.isna().sum(),
    "Porcentaje": df_browse.isna().mean() * 100,
})

missing_table = missing_table.sort_values(by="Porcentaje", ascending=False)
missing_table

,Valores_Faltantes,Porcentaje
member_id,2260701,100.000000
verification_status_joint,2144971,94.880791
dti_joint,2139995,94.660683
annual_inc_joint,2139991,94.660506
desc,2134636,94.423632
mths_since_last_record,1901545,84.113069
mths_since_last_major_derog,1679926,74.309960
next_pymnt_d,1345343,59.509993
mths_since_last_delinq,1158535,51.246715
il_util,1068883,47.281042


In [133]:
duplicados = df_browse.duplicated().sum()

print(f"Filas duplicadas: {duplicados}")

Filas duplicadas: 0
